In [0]:
%pip install databricks-sdk==0.103.0

In [0]:
%restart_python

In [0]:
%run ../../Includes/Classroom-Setup-Common-Python

In [0]:
# import json
# from databricks.sdk import WorkspaceClient

# w = WorkspaceClient()
# space_id = "01f14e1f9dce1829a9de4672a864300f"

# resp = w.api_client.do(
#     "GET",
#     f"/api/2.0/genie/spaces/{space_id}?include_serialized_space=true",
# )
# print(json.dumps(json.loads(resp["serialized_space"]), indent=2))

In [0]:
"""
Bakehouse Genie Space — single-file backup script.

LAYOUT
------
1. create_genie_space()      -- reusable function (works for any course)
2. Bakehouse content         -- module-level variables
3. build_bakehouse_space()   -- calls #1 with #2

NOTEBOOK USAGE
--------------
    my_catalog = "labuser_jane_doe"   # set this FIRST
    %run ./bakehouse_genie_space
    space_id = build_bakehouse_space()

`my_catalog` MUST be defined in the notebook before %run -- the variables
below interpolate it at module-load time.

Backed by the documented endpoint POST /api/2.0/genie/spaces.
"""

import json
import uuid
from typing import Optional

from databricks.sdk import WorkspaceClient


# =====================================================================
# 1. Reusable function: create a Genie Space from any inputs
# =====================================================================

def create_genie_space(
    *,
    title: str,
    description: str,
    table_identifiers: list[str],
    warehouse_name: str = "shared_warehouse",
    sample_questions: Optional[list[str]] = None,
    general_instructions: Optional[str] = None,
    sql_examples: Optional[list[dict]] = None,
    sql_expressions: Optional[list[dict]] = None,
    benchmarks: Optional[list[dict]] = None,
) -> str:
    """Create a Genie Space and return its space_id.

    `sql_expressions` are custom dimensions / metrics (CASE statements,
    calculated columns) shown under Knowledge Store -> SQL Expressions.
    Each item: {"name": str, "sql": str, "instruction": str (optional),
    "synonyms": list[str] (optional)}.
    """
    def hex_id() -> str:
        return uuid.uuid4().hex

    def lines(text: str) -> list[str]:
        return text.splitlines(keepends=True)

    print(f"[setup]    Building Genie Space '{title}'")
    print(
        f"[setup]    {len(table_identifiers)} tables, "
        f"{len(sample_questions or [])} sample questions, "
        f"{len(sql_examples or [])} SQL examples, "
        f"{len(sql_expressions or [])} SQL expressions, "
        f"{len(benchmarks or [])} benchmarks"
    )

    # ---- Build serialized_space JSON ----
    payload: dict = {
        "version": 2,
        "data_sources": {
            "tables": sorted(
                [{"identifier": t} for t in table_identifiers],
                key=lambda x: x["identifier"],
            ),
        },
    }

    if sample_questions:
        payload["config"] = {
            "sample_questions": [
                {"id": hex_id(), "question": [q]} for q in sample_questions
            ],
        }

    instructions: dict = {}
    if general_instructions:
        instructions["text_instructions"] = [
            {"id": hex_id(), "content": lines(general_instructions)},
        ]
    if sql_examples:
        instructions["example_question_sqls"] = [
            {"id": hex_id(), "question": [e["question"]], "sql": lines(e["sql"])}
            for e in sql_examples
        ]
    if sql_expressions:
        instructions["sql_snippets"] = {
            "expressions": [
                {
                    "id": hex_id(),
                    "display_name": e["name"],
                    "sql": lines(e["sql"]),
                    **({"instruction": lines(e["instruction"])} if e.get("instruction") else {}),
                    **({"synonyms": list(e["synonyms"])} if e.get("synonyms") else {}),
                }
                for e in sql_expressions
            ],
        }
    if instructions:
        payload["instructions"] = instructions

    if benchmarks:
        payload["benchmarks"] = {
            "questions": [
                {
                    "id": hex_id(),
                    "question": [b["question"]],
                    "answer": [{"format": "SQL", "content": lines(b["sql"])}],
                }
                for b in benchmarks
            ],
        }

    serialized_space = json.dumps(payload)

    # ---- Resolve warehouse ----
    w = WorkspaceClient()

    FREE_EDITION = "Serverless Starter Warehouse"
    warehouses = list(w.warehouses.list())
    warehouse_id = None
    for candidate in (warehouse_name, FREE_EDITION):
        matches = [wh for wh in warehouses if wh.name == candidate]
        if len(matches) == 1:
            warehouse_id = matches[0].id
            break
        if len(matches) > 1:
            raise ValueError(f"Multiple warehouses named {candidate!r}.")
    if not warehouse_id:
        available = ", ".join(sorted(wh.name or "" for wh in warehouses)) or "(none)"
        raise ValueError(
            f"No warehouse named {warehouse_name!r} or {FREE_EDITION!r}. "
            f"Available: {available}"
        )
    resolved_name = next(wh.name for wh in warehouses if wh.id == warehouse_id)
    print(f"[setup]    Warehouse: '{resolved_name}' ({warehouse_id})")

    # ---- Create the Space ----
    parent_path = f"/Workspace/Users/{w.current_user.me().user_name}"
    print(f"[setup]    Parent path: {parent_path}")

    space = w.genie.create_space(
        warehouse_id=warehouse_id,
        serialized_space=serialized_space,
        title=title,
        description=description,
        parent_path=parent_path,
    )
    space_id = getattr(space, "space_id", None) or getattr(space, "id", None)
    if not space_id:
        raise RuntimeError(f"Create response missing space_id: {space}")
    print(f"[create] Genie Space '{title}' -> {space_id}")
    return space_id


In [0]:
SPACE_INSTRUCTIONS = """
- Formatting
    - Format all results using US dollars
    - Round all currency values to 2 decimal places.

- Business Rules
    - `totalPrice` represents both revenue and sales volume
    - Always include franchise name with franchise ID
    - Use `totalPrice` for total sales and performance comparisons

- Fiscal Calendar
    - Fiscal year starts in February
    - Q1: February, March, April
    - Q2: May, June, July
    - Q3: August, September, October
    - Q4: November, December, January
"""

In [0]:
SAMPLE_QUESTIONS = [
    "How many customers do we have?"
]

In [0]:
SQL_EXPRESSIONS = [
    {
        "name": "region",
        "sql": f"""CASE
    WHEN {my_catalog}.genie_course_bakehouse.sales_franchises_gold.country IN ('Japan', 'Australia')
      THEN 'APJ'
    WHEN {my_catalog}.genie_course_bakehouse.sales_franchises_gold.country IN ('US', 'Canada')
      THEN 'AMER'
    WHEN {my_catalog}.genie_course_bakehouse.sales_franchises_gold.country IN ('Netherlands', 'France', 'Germany', 'Italy', 'Sweden')
      THEN 'EMEA'
    ELSE 'Other'
END""",
        "instruction": (
            "When user asks for metrics by 'region' where region is defined by "
            "business mapping of country to APJ, AMER, EMEA, or Other; "
            "SCOPE_TABLES: Use with sales_franchises_gold and sales_transactions_gold, "
            "grouping or reporting at the franchise-location level; "
            "RISK_IF_MISUSED: Do not use unless the user explicitly refers to regions "
            "per this mapping (e.g., could mislead if a different regional breakdown is "
            "implied). This is a company-specific mapping."
        ),
        "synonyms": ["franchise region", "region sales"],
    },
]

In [0]:
SQL_EXAMPLES = [
    {"question": "Who are our best customers?", "sql": f"""
        SELECT
            c.customerID,
            c.first_name,
            c.last_name,
            SUM(t.totalPrice) AS total_spend
        FROM {my_catalog}.genie_course_bakehouse.sales_transactions_gold AS t
            JOIN {my_catalog}.genie_course_bakehouse.sales_customers_gold AS c
                ON t.customerID = c.customerID
        GROUP BY c.customerID, c.first_name, c.last_name
        ORDER BY total_spend DESC
        LIMIT 10
    """},
    {"question": "How many franchises does each supplier serve?", "sql": f"""
        SELECT
            COALESCE(s.name, 'No Matching Supplier') AS supplier_name,
            COUNT(f.franchiseID) AS franchise_count
        FROM {my_catalog}.genie_course_bakehouse.sales_franchises_gold AS f
            LEFT JOIN {my_catalog}.genie_course_bakehouse.sales_suppliers_gold AS s
                ON f.supplierID = s.supplierID
        GROUP BY s.name
        ORDER BY franchise_count DESC
    """},
]

In [0]:
BENCHMARKS = [
    {"question": "How many customers do we have?", "sql": f"""
        SELECT COUNT(*) AS total_customers
        FROM {my_catalog}.genie_course_bakehouse.sales_customers_gold
    """},
    {"question": "What are the top 5 franchise locations by total revenue?", "sql": f"""
        WITH franchise_revenue AS (
            SELECT
                f.franchiseID,
                f.city,
                f.name,
                SUM(t.totalPrice) AS total_revenue
            FROM {my_catalog}.genie_course_bakehouse.sales_transactions_gold t
                JOIN {my_catalog}.genie_course_bakehouse.sales_franchises_gold f
                    ON t.franchiseID = f.franchiseID
            WHERE t.franchiseID IS NOT NULL
                AND t.totalPrice IS NOT NULL
                AND f.city IS NOT NULL
            GROUP BY f.franchiseID, f.city, f.name
        )
        SELECT city, name, total_revenue
        FROM (
            SELECT
                city,
                name,
                total_revenue,
                RANK() OVER (ORDER BY total_revenue DESC) AS rank
            FROM franchise_revenue
        )
        WHERE rank <= 5
    """},
    {"question": "How many franchises does each supplier serve?", "sql": f"""
        SELECT
            COALESCE(s.name, 'No Matching Supplier') AS supplier_name,
            COUNT(f.franchiseID) AS franchise_count
        FROM {my_catalog}.genie_course_bakehouse.sales_franchises_gold AS f
            LEFT JOIN {my_catalog}.genie_course_bakehouse.sales_suppliers_gold AS s
                ON f.supplierID = s.supplierID
        GROUP BY s.name
        ORDER BY franchise_count DESC
    """},
    {"question": "Who are our best customers?", "sql": f"""
        SELECT
            c.customerID,
            c.first_name,
            c.last_name,
            SUM(t.totalPrice) AS total_spend
        FROM {my_catalog}.genie_course_bakehouse.sales_transactions_gold AS t
            JOIN {my_catalog}.genie_course_bakehouse.sales_customers_gold AS c
                ON t.customerID = c.customerID
        GROUP BY c.customerID, c.first_name, c.last_name
        ORDER BY total_spend DESC
        LIMIT 10
    """},
    {"question": "What are total sales by region?", "sql": f"""
        SELECT
            CASE
                WHEN f.country IN ('Japan', 'Australia') THEN 'APJ'
                WHEN f.country IN ('US', 'Canada') THEN 'AMER'
                WHEN f.country IN ('Netherlands', 'France', 'Germany', 'Italy', 'Sweden') THEN 'EMEA'
                ELSE 'Other'
            END AS region,
            SUM(t.totalPrice) AS total_sales
        FROM {my_catalog}.genie_course_bakehouse.sales_franchises_gold AS f
            JOIN {my_catalog}.genie_course_bakehouse.sales_transactions_gold AS t
                ON f.franchiseID = t.franchiseID
        GROUP BY region
        ORDER BY total_sales DESC
    """},
    {"question": "Count of bad reviews by locations?", "sql": f"""
        SELECT
            r.ref,
            f.name AS franchise_name,
            f.city,
            COUNT(*) AS negative_review_count
        FROM {my_catalog}.genie_course_bakehouse.media_customer_reviews_gold r
            JOIN {my_catalog}.genie_course_bakehouse.sales_franchises_gold f
                ON r.ref = f.franchiseID
        WHERE r.flag = 'negative'
        GROUP BY r.ref, f.name, f.city
        ORDER BY negative_review_count DESC
    """},
]

In [0]:
# =====================================================================
# TABLES
# =====================================================================


TABLE_IDENTIFIERS = [
    f"{my_catalog}.genie_course_bakehouse.sales_customers_gold",
    f"{my_catalog}.genie_course_bakehouse.sales_franchises_gold",
    f"{my_catalog}.genie_course_bakehouse.sales_suppliers_gold",
    f"{my_catalog}.genie_course_bakehouse.sales_transactions_gold",
    f"{my_catalog}.genie_course_bakehouse.media_customer_reviews_gold",
]

